In [ ]:
import re
import time
import json
import requests
from pathlib import Path
from tqdm import tqdm

HEADERS_SEC = {
    "User-Agent": "taithientam2k4@gmail.com",
    "Accept-Encoding": "gzip, deflate",
}

RAW_DIR = Path("data/raw_filings")
RAW_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FORMS = {"10-K", "10-Q", "8-K", "DEF 14A"}


In [ ]:

def safe_name(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", text)

def get_ticker_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    r = requests.get(url, headers=HEADERS_SEC, timeout=30)
    r.raise_for_status()

    data = r.json()
    ticker_map = {}

    for _, item in data.items():
        ticker = item["ticker"].upper()
        cik = str(item["cik_str"]).zfill(10)
        company = item["title"]
        ticker_map[ticker] = {
            "cik": cik,
            "company": company
        }

    return ticker_map

def get_submissions(cik: str):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS_SEC, timeout=30)
    r.raise_for_status()
    return r.json()

def build_filing_url(cik: str, accession: str, primary_doc: str):
    cik_no_zero = str(int(cik))
    accession_no_dash = accession.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_no_zero}/{accession_no_dash}/{primary_doc}"

def crawl_ticker(ticker: str, max_files: int = 10):
    ticker = ticker.upper()
    ticker_map = get_ticker_cik_map()

    if ticker not in ticker_map:
        print(f"Không tìm thấy ticker: {ticker}")
        return

    cik = ticker_map[ticker]["cik"]
    company = ticker_map[ticker]["company"]

    print(f"Crawl {ticker} - {company} - CIK {cik}")

    submissions = get_submissions(cik)
    recent = submissions["filings"]["recent"]

    count = 0

    for form, accession, filing_date, primary_doc in zip(
        recent["form"],
        recent["accessionNumber"],
        recent["filingDate"],
        recent["primaryDocument"]
    ):
        if form not in TARGET_FORMS:
            continue

        url = build_filing_url(cik, accession, primary_doc)

        out_dir = RAW_DIR / ticker / form
        out_dir.mkdir(parents=True, exist_ok=True)

        filename = safe_name(f"{ticker}_{form}_{filing_date}_{accession}_{primary_doc}")
        out_path = out_dir / filename

        try:
            r = requests.get(url, headers=HEADERS_SEC, timeout=60)
            r.raise_for_status()

            out_path.write_bytes(r.content)

            metadata = {
                "ticker": ticker,
                "company": company,
                "cik": cik,
                "form": form,
                "filing_date": filing_date,
                "accession": accession,
                "primary_doc": primary_doc,
                "url": url,
                "local_path": str(out_path)
            }

            meta_path = out_path.with_suffix(out_path.suffix + ".json")
            meta_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

            print("Saved:", out_path)

            count += 1
            time.sleep(0.2)  

            if count >= max_files:
                break

        except Exception as e:
            print("Error:", ticker, form, accession, e)

def main():
    tickers = ["MS", "HD", "INTU", "AAPL", "NVDA"]

    for ticker in tickers:
        crawl_ticker(ticker, max_files=10)



In [3]:
if __name__ == "__main__":
    main()

Crawl MS - MORGAN STANLEY - CIK 0000895421
Saved: data\raw_filings\MS\8-K\MS_8-K_2026-04-15_0000895421-26-000111_ms-20260415.htm
Saved: data\raw_filings\MS\DEF 14A\MS_DEF_14A_2026-04-02_0001140361-26-012975_ny20058185x1_def14a.htm
Saved: data\raw_filings\MS\10-K\MS_10-K_2026-02-19_0000895421-26-000086_ms-20251231.htm
Saved: data\raw_filings\MS\8-K\MS_8-K_2026-02-11_0000950103-26-001967_dp240377_8k.htm
Saved: data\raw_filings\MS\8-K\MS_8-K_2026-01-15_0000895421-26-000007_ms-20260115.htm
Saved: data\raw_filings\MS\10-Q\MS_10-Q_2025-11-03_0000895421-25-000539_ms-20250930.htm
Saved: data\raw_filings\MS\8-K\MS_8-K_2025-10-15_0000895421-25-000535_ms-20251015.htm
Saved: data\raw_filings\MS\8-K\MS_8-K_2025-09-30_0000950103-25-012581_dp235215_8k.htm
Saved: data\raw_filings\MS\8-K\MS_8-K_2025-09-23_0000950103-25-012045_dp234752_8k.htm
Saved: data\raw_filings\MS\10-Q\MS_10-Q_2025-08-04_0000895421-25-000440_ms-20250630.htm
Crawl HD - HOME DEPOT, INC. - CIK 0000354950
Saved: data\raw_filings\HD\DEF